# Vitalia HAR — UT Complex Dataset (LSTM, Kaggle)

Adaptación del notebook original de [Human-Activity-Recognition-Keras-Android](https://github.com/dspanah/Human-Activity-Recognition-Keras-Android) usando el dataset **UT_Data_Complex** filtrado a las 5 clases Vitalia.

**Dataset a añadir:** `ut-data-complex` con los ficheros `smartphoneatpocket.csv` y `smartphoneatwrist.csv`.

**Clases filtradas (de las 13 originales):**
- 11111 walk → 1 walking
- 11112 stand → 0 static
- 11113 jog → 2 running
- 11114 sit → 0 static
- 11116 upstairs → 3 upstairs
- 11117 downstairs → 4 downstairs

**Output (`/kaggle/working/`):** `har_ut_model_int8.tflite`, `har_ut_model_fp16.tflite`, `har_ut_model.keras`, `model_meta_ut.json`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, json, math, warnings
from pathlib import Path
from collections import Counter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from scipy import stats

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

WORKING_DIR = Path('/kaggle/working')
INPUT_DIR   = Path('/kaggle/input')

print(f'TensorFlow {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────

# Kaggle dataset slug for UT_Data_Complex files
# Add dataset 'ut-data-complex' containing smartphoneatpocket.csv + smartphoneatwrist.csv
_dataset_candidates = [
    INPUT_DIR / 'datasets' / 'jaimenovillo' / 'ut-data-complex',
    INPUT_DIR / 'ut-data-complex',
    INPUT_DIR / 'ut-complex',
]
DATA_DIR = next((p for p in _dataset_candidates if p.exists()), None)
print(f'DATA_DIR: {DATA_DIR}')

# UT_Data_Complex: 13 activity label codes (no header in CSV)
# We keep only the 6 Vitalia-relevant ones; sit+stand merge into static.
# Format: {original_code: canonical_index}
LABEL_FILTER = {
    11111: 1,  # walk     → walking
    11112: 0,  # stand    → static
    11113: 2,  # jog      → running
    11114: 0,  # sit      → static
    11116: 3,  # upstairs → upstairs
    11117: 4,  # downstairs → downstairs
}

N_CLASSES   = 5
CLASS_NAMES = ['static', 'walking', 'running', 'upstairs', 'downstairs']

# Sliding window — same as original notebook
N_TIME_STEPS = 100
STEP         = 50
N_FEATURES   = 12  # Ax,Ay,Az,Lx,Ly,Lz,Gx,Gy,Gz,MA,ML,MG

# Training
BATCH_SIZE = 1024
N_EPOCHS   = 30

print(f'Classes ({N_CLASSES}): {CLASS_NAMES}')
print(f'Window: {N_TIME_STEPS} steps, stride {STEP}')

## Loading and preparing train and test data

Loads both pocket and wrist CSV files from UT_Data_Complex, filters to Vitalia classes,
computes magnitude features, and stacks both sensor positions (doubles training data).

Column layout of each CSV (no header):
`timestamp, Ax, Ay, Az, Lx, Ly, Lz, Gx, Gy, Gz, Mx, My, Mz, activity_label`

In [ ]:
# UT_Data_Complex CSV column names (no header in file)
COL_NAMES = [
    'timestamp',
    'acc_x', 'acc_y', 'acc_z',
    'lin_x', 'lin_y', 'lin_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'mag_x', 'mag_y', 'mag_z',
    'label',
]

def load_ut_csv(path):
    """Load a UT_Data_Complex CSV, filter to Vitalia classes, remap labels."""
    print(f'  Reading {Path(path).name}...')
    df = pd.read_csv(path, header=None, names=COL_NAMES)
    df = df[df['label'].isin(LABEL_FILTER.keys())].copy()
    df['label'] = df['label'].map(LABEL_FILTER)
    return df


def add_magnitude_features(df):
    """Add magnitude features MA, ML, MG — same as original notebook."""
    df = df.copy()
    df['MA'] = np.sqrt(df['acc_x']**2  + df['acc_y']**2  + df['acc_z']**2)
    df['ML'] = np.sqrt(df['lin_x']**2  + df['lin_y']**2  + df['lin_z']**2)
    df['MG'] = np.sqrt(df['gyro_x']**2 + df['gyro_y']**2 + df['gyro_z']**2)
    return df


# ── Load pocket + wrist and concatenate (mimics original left+right pocket doubling) ──
if DATA_DIR is None:
    raise RuntimeError('Dataset not found. Add kaggle dataset ut-data-complex with the two CSV files.')

pocket_path = DATA_DIR / 'smartphoneatpocket.csv'
wrist_path  = DATA_DIR / 'smartphoneatwrist.csv'

dfs = []
for p in [pocket_path, wrist_path]:
    if p.exists():
        d = load_ut_csv(p)
        d = add_magnitude_features(d)
        dfs.append(d)
        print(f'    {len(d):,} rows after filtering')
    else:
        print(f'  WARNING: {p.name} not found — skipping')

if not dfs:
    raise RuntimeError('No CSV files found in DATA_DIR')

df = pd.concat(dfs, ignore_index=True)
print(f'\nTotal rows: {len(df):,}')
print('Class distribution (canonical labels):')
for lbl, name in enumerate(CLASS_NAMES):
    n = (df['label'] == lbl).sum()
    print(f'  {lbl} {name:12s}: {n:,}')
df.head()

## 80/20 train/test split

In [ ]:
# 80/20 split on the full DataFrame (same approach as original notebook)
split_point = int(len(df) * 0.8)
train_data = df.iloc[:split_point].reset_index(drop=True)
test_data  = df.iloc[split_point:].reset_index(drop=True)

FEATURE_COLS = ['acc_x','acc_y','acc_z','lin_x','lin_y','lin_z',
                'gyro_x','gyro_y','gyro_z','MA','ML','MG']

train_X_raw = train_data[FEATURE_COLS].values.astype(np.float32)
train_y_raw = train_data['label'].values
test_X_raw  = test_data[FEATURE_COLS].values.astype(np.float32)
test_y_raw  = test_data['label'].values

print(f'Train: {len(train_X_raw):,}  Test: {len(test_X_raw):,}')
print(f'Features: {train_X_raw.shape[1]}')
train_data[FEATURE_COLS].head()

## Sliding window mechanism

Exact same approach as original notebook: windows of 100 timesteps with stride 50.
Label = mode of the window.

In [ ]:
def generate_sequence(x_data, y_data, n_time_steps=N_TIME_STEPS, step=STEP):
    """Sliding window — identical to original notebook's generate_sequence()."""
    segments, labels = [], []
    for start in range(0, len(x_data) - n_time_steps, step):
        end = start + n_time_steps
        window = x_data[start:end]
        label  = stats.mode(y_data[start:end], keepdims=True)[0][0]
        segments.append(window)
        labels.append(label)
    return np.array(segments, dtype=np.float32), np.array(labels, dtype=np.int32)


def reshape_segments(segments, labels, n_classes=N_CLASSES):
    """Reshape to (N, time_steps, features) and one-hot encode labels."""
    x = segments.reshape(-1, N_TIME_STEPS, N_FEATURES)
    y = keras.utils.to_categorical(labels, num_classes=n_classes).astype(np.float32)
    return x, y


print('Generating sliding windows...')
train_seg, train_lbl = generate_sequence(train_X_raw, train_y_raw)
test_seg,  test_lbl  = generate_sequence(test_X_raw,  test_y_raw)

train_X, train_y = reshape_segments(train_seg, train_lbl)
test_X,  test_y  = reshape_segments(test_seg,  test_lbl)

print(f'train_X: {train_X.shape}  train_y: {train_y.shape}')
print(f'test_X:  {test_X.shape}   test_y:  {test_y.shape}')

## Building the model

Same architecture as original: LSTM(32) → Flatten → Dense(32) → Dense(n_classes).
L2 regularization on all layers.

In [ ]:
L2 = 0.000001

model = keras.Sequential([
    layers.LSTM(
        32,
        input_shape=(N_TIME_STEPS, N_FEATURES),
        return_sequences=True,
        kernel_initializer='orthogonal',
        kernel_regularizer=regularizers.l2(L2),
        recurrent_regularizer=regularizers.l2(L2),
        bias_regularizer=regularizers.l2(L2),
        name='LSTM_1',
    ),
    layers.Flatten(name='Flatten'),
    layers.Dense(
        32, activation='relu',
        kernel_regularizer=regularizers.l2(L2),
        bias_regularizer=regularizers.l2(L2),
        name='Dense_1',
    ),
    layers.Dense(
        N_CLASSES, activation='softmax',
        kernel_regularizer=regularizers.l2(L2),
        bias_regularizer=regularizers.l2(L2),
        name='Dense_2',
    ),
], name='HAR_LSTM')

model.summary()

## Training & Evaluation

In [ ]:
model.compile(
    optimizer=keras.optimizers.RMSprop(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

history = model.fit(
    train_X, train_y,
    batch_size=BATCH_SIZE,
    epochs=N_EPOCHS,
    validation_data=(test_X, test_y),
    verbose=1,
)

train_loss, train_acc = model.evaluate(train_X, train_y, verbose=0)
test_loss,  test_acc  = model.evaluate(test_X,  test_y,  verbose=0)
print(f'\nTrain accuracy: {train_acc:.4f}')
print(f'Test  accuracy: {test_acc:.4f}')

## Confusion Matrix

In [ ]:
y_pred = model.predict(test_X, verbose=0)
y_true_idx = test_y.argmax(axis=1)
y_pred_idx = y_pred.argmax(axis=1)

print(classification_report(y_true_idx, y_pred_idx, target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(y_true_idx, y_pred_idx)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'LSTM HAR — UT Complex ({N_CLASSES} classes) | acc={test_acc:.3f}')
plt.tight_layout()
plt.savefig(WORKING_DIR / 'confusion_matrix_ut.png', dpi=120)
plt.show()

# Training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss'); axes[1].legend()
plt.tight_layout()
plt.savefig(WORKING_DIR / 'training_history_ut.png', dpi=120)
plt.show()

## Exporting the model

Saves Keras model + TFLite INT8 + FP16 to `/kaggle/working/`.

In [ ]:
# ── Save Keras model ──────────────────────────────────────────────────────────
keras_path = WORKING_DIR / 'har_ut_model.keras'
model.save(str(keras_path))
print(f'Keras model saved: {keras_path}')

# ── TFLite FP16 (always works with LSTM) ─────────────────────────────────────
converter_fp16 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_fp16.optimizations = [tf.lite.Optimize.DEFAULT]
converter_fp16.target_spec.supported_types = [tf.float16]
tflite_fp16 = converter_fp16.convert()
fp16_path = WORKING_DIR / 'har_ut_model_fp16.tflite'
fp16_path.write_bytes(tflite_fp16)
print(f'FP16: {fp16_path} ({len(tflite_fp16)/1024:.1f} KB)')

# ── TFLite INT8 (full quantization) ─────────────────────────────────────────
try:
    converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
    converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]

    def representative_dataset():
        idx = np.random.choice(len(train_X), size=min(400, len(train_X)), replace=False)
        for i in idx:
            yield [train_X[i:i+1]]

    converter_int8.representative_dataset = representative_dataset
    converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter_int8.inference_input_type  = tf.int8
    converter_int8.inference_output_type = tf.int8
    tflite_int8 = converter_int8.convert()
    int8_path = WORKING_DIR / 'har_ut_model_int8.tflite'
    int8_path.write_bytes(tflite_int8)
    print(f'INT8: {int8_path} ({len(tflite_int8)/1024:.1f} KB)')
except Exception as e:
    print(f'INT8 conversion failed ({e}). Use FP16 model.')
    tflite_int8 = None

# ── Verify FP16 output shape ──────────────────────────────────────────────────
interp = tf.lite.Interpreter(model_content=tflite_fp16)
interp.allocate_tensors()
in_det  = interp.get_input_details()[0]
out_det = interp.get_output_details()[0]
print(f'FP16 input:  {in_det["shape"]} dtype={in_det["dtype"]}')
print(f'FP16 output: {out_det["shape"]} dtype={out_det["dtype"]}')
assert out_det['shape'][1] == N_CLASSES, f'Output mismatch: {out_det["shape"][1]} != {N_CLASSES}'

In [ ]:
# ── Save model metadata ────────────────────────────────────────────────────────
meta = {
    'class_names': CLASS_NAMES,
    'n_classes': N_CLASSES,
    'window_size': N_TIME_STEPS,
    'step': STEP,
    'n_features': N_FEATURES,
    'features': ['acc_x','acc_y','acc_z','lin_x','lin_y','lin_z',
                 'gyro_x','gyro_y','gyro_z','MA','ML','MG'],
    'architecture': 'LSTM(32) → Flatten → Dense(32,relu) → Dense(n_classes,softmax)',
    'dataset': 'UT_Data_Complex (pocket + wrist)',
    'test_accuracy': round(float(test_acc), 4),
    'cycling_trained': False,
    'vitapoints_per_min': {'static':0,'walking':2,'running':5,'upstairs':3,'downstairs':2},
    'fp16_size_kb': round(len(tflite_fp16)/1024, 1),
    'int8_size_kb': round(len(tflite_int8)/1024, 1) if tflite_int8 else None,
}
(WORKING_DIR / 'model_meta_ut.json').write_text(json.dumps(meta, indent=2))

print('\n=== VITALIA HAR — UT Complex (LSTM) ===')
print(f'Classes:       {", ".join(CLASS_NAMES)}')
print(f'Test accuracy: {test_acc:.4f}')
print(f'Input shape:   (1, {N_TIME_STEPS}, {N_FEATURES})')
print('\nFiles in /kaggle/working/:')
for f in sorted(WORKING_DIR.iterdir()):
    print(f'  {f.name} ({f.stat().st_size/1024:.1f} KB)')

## Notas de integración con Flutter

**IMPORTANTE:** Este modelo tiene una arquitectura diferente al ResNet1D actual de la app:
- Input: `(1, 100, 12)` — 100 timesteps × 12 features (vs. 128×6 del ResNet1D)
- Features: acelerómetro + aceleración lineal + giroscopio + magnitudes (vs. solo lineal+giroscopio)
- Clases: 5 (sin cycling)

Para usar este modelo en Flutter, hay que actualizar:
1. `har_classifier.dart`: `windowSize = 100`, `channels = 12`, enum Activity (5 clases)
2. `sensor_service.dart`: añadir canal de magnitudes MA/ML/MG
3. `SlidingWindowBuffer`: window=100, step=50

**Alternativa recomendada para comparar:** correr ambos modelos y ver cuál da mejor accuracy en el dispositivo real.